# FreeLLMAPI in Google Colab — persistente Umgebung

Dieses Notebook startet [FreeLLMAPI](https://github.com/tashfeenahmed/freellmapi) auf der
temporären Colab-VM und legt alle wichtigen Daten **verschlüsselt in Google Drive** ab, damit
eine neue Colab-VM dieselbe Installation (inklusive aller Provider-Keys) wieder hochzieht.

**Zielarchitektur**

```
Google Colab
   └── temporäre Linux-VM
          └── FreeLLMAPI  (Node.js, ohne Docker)
                 ├── Dashboard      http://127.0.0.1:3001/
                 └── OpenAI-kompatible API   http://127.0.0.1:3001/v1
```

Kein Ollama, keine lokale Inferenz, keine GPU, kein Tunnel, kein zusätzliches KI-Framework.

---

## Was dieses Notebook verwendet (alles aus der aktuellen Doku belegt)

| Thema | Verwendeter Wert | Quelle im FreeLLMAPI-Repo (Tag `v0.9.5`) |
|---|---|---|
| Installationsweg | `npm run build` → `node server/dist/index.js` | `docs/install.md`, Abschnitt *„For a production build without Docker"* |
| Port | `PORT=3001` (dokumentierter Default) | `docs/env/01-variables.md` |
| Bindung | `HOST=127.0.0.1` (nur lokal) | `docs/env/01-variables.md` |
| Datenbankpfad | `FREEAPI_DB_PATH` | `docs/env/01-variables.md`, `docs/install.md` |
| Verschlüsseltes Backup | `FREEAPI_DB_BACKUP_PATH`, `FREEAPI_DB_BACKUP_INTERVAL_MS` | `docs/deployment/02-updates-and-backup.md` |
| Backup-Schlüssel | `FREEAPI_DB_BACKUP_KEY`, Default = `ENCRYPTION_KEY` | `docs/env/01-variables.md`, `server/src/lib/db-backup.ts` |
| Schlüssel | `ENCRYPTION_KEY`, exakt 64 Hex-Zeichen | `docs/env/02-security-and-keys.md` |
| Verzeichnis-Härtung | `FREEAPI_DB_DIR_HARDENING=1` | `docs/env/01-variables.md` |
| Health-Endpunkt | `GET /api/ping` (ohne Auth) | `server/src/app.ts`, Healthcheck im `Dockerfile` |
| Model-Endpunkt | `GET /v1/models` (mit Unified Key) | `docs/api/01-rest-api.md` |

> **Wichtig:** Docker ist laut FreeLLMAPI zwar der empfohlene Installationsweg, auf einer
> **verwalteten Colab-Runtime aber nicht nutzbar** — es gibt keinen Docker-Daemon, und die
> [Colab-FAQ](https://research.google.com/colaboratory/faq.html) listet „employing techniques
> such as containerization to circumvent anti-abuse policies" als untersagt. Deshalb nutzt
> dieses Notebook den offiziell dokumentierten Weg **ohne** Docker.

---

## ⚠️ Zwei Dinge, die du vorher wissen musst

1. **Colab-Nutzungsrichtlinien.** Die Colab-FAQ verbietet auf verwalteten Runtimes u. a.
   *„file hosting, media serving, or other web service offerings not related to interactive
   compute"* und — auf der kostenlosen Stufe — *„bypassing the notebook UI to interact primarily
   via a web UI"*. Ein dauerhaft laufender LLM-Gateway mit Web-Dashboard fällt genau in diese
   Grauzone. **Die Runtime kann jederzeit beendet werden.** Für Dauerbetrieb ist ein eigener
   Host (VPS, Raspberry Pi, Docker) der richtige Ort; Colab taugt hier zum Ausprobieren.
2. **Maximale Laufzeit.** Laut Colab-FAQ laufen kostenlose Notebooks *„for at most 12 hours"*,
   idle Runtimes werden früher gelöscht. Genau dafür ist die Drive-Persistenz da.

---

## Einmalige Vorbereitung

1. **Colab-Secret anlegen** — linke Seitenleiste → 🔑 *Secrets* → *Add a new secret*:
   - **Name:** `FREELLMAPI_ENCRYPTION_KEY`
   - **Wert:** 64 Hex-Zeichen, erzeugt mit `openssl rand -hex 32`
   - Schalter **„Notebook access"** aktivieren
2. **Zellen 1–8 der Reihe nach ausführen** (Laufzeit → Alle Zellen ausführen).
3. Beim **ersten Start**: Dashboard öffnen, Account anlegen, unter **Keys** die Provider-Keys
   eintragen. Danach liegen sie **AES-256-GCM-verschlüsselt** in der SQLite-Datenbank.

**Optionale Secrets**

| Name | Zweck |
|---|---|
| `FREELLMAPI_DB_BACKUP_KEY` | Getrennter 64-Hex-Schlüssel nur für die Backup-Hülle. Fehlt er, nutzt FreeLLMAPI `ENCRYPTION_KEY` (dokumentierter Default). |
| `FREELLMAPI_UNIFIED_KEY` | Der `freellmapi-…`-Schlüssel aus dem Dashboard. Nur dafür, damit der Abschlusstest `/v1/models` mit HTTP 200 statt 401 prüfen kann. |

**Reihenfolge der Zellen**

| Zelle | Inhalt |
|---|---|
| 1 | Konfiguration · Drive mounten · Secrets laden · Persistenz anlegen |
| 2 | Systemprüfung (OS, Python, Node, npm, RAM, CPU, Disk, Docker) |
| 3 | Installation (mit Build-Cache in Drive) |
| 4 | Persistente Datenbank + Backup-Schutz |
| 5 | FreeLLMAPI starten |
| 6 | Health-Check + Dashboard öffnen |
| 7 | Abschlusstest |
| 8 | Stoppen |


## Zelle 1 — Start / Voraussetzungen

Google Drive mounten, Colab-Secret lesen (ohne es auszugeben), Persistenz-Ordner anlegen und
erkennen, ob bereits Daten vorhanden sind.

In [ ]:
# ============================================================================
#  ZELLE 1 — KONFIGURATION + VORAUSSETZUNGEN
# ============================================================================
import os
import re
import json
import time
import shlex
import shutil
import signal
import platform
import subprocess
import urllib.error
import urllib.request
from pathlib import Path

# ---------------------------- Konfiguration ---------------------------------
FREELLMAPI_REF     = "v0.9.5"      # gepinnter Release-Tag -> reproduzierbar
REPO_URL           = "https://github.com/tashfeenahmed/freellmapi.git"
PORT               = 3001          # dokumentierter Default-Port (docs/env/01-variables.md)
HOST               = "127.0.0.1"   # nur lokal erreichbar, kein Tunnel
BACKUP_INTERVAL_MS = 120000        # dokumentierter Default ist 300000 (5 min);
                                   # 2 min = weniger Datenverlust bei abruptem VM-Ende

APP_DIR   = Path("/content/freellmapi")          # flüchtig (Colab-VM)
DB_DIR    = APP_DIR / "data"
DB_PATH   = DB_DIR / "freeapi.db"
LOG_DIR   = APP_DIR / "logs"
LOG_FILE  = LOG_DIR / "server.log"
PID_FILE  = APP_DIR / "server.pid"
REF_MARK  = APP_DIR / ".freellmapi-ref"

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT  = DRIVE_MOUNT / "My Drive" / "FreeLLMAPI"   # dauerhaft
BACKUP_FILE = DRIVE_ROOT / "freeapi.db.backup"          # verschlüsselt (FreeLLMAPI)
BACKUP_PREV = DRIVE_ROOT / "freeapi.db.backup.prev"     # Rollback-Kopie (dieses Notebook)
APP_CACHE   = DRIVE_ROOT / ("app-cache-%s.tar.gz" % FREELLMAPI_REF)  # nur Code, keine Secrets

SECRET_ENCRYPTION = "FREELLMAPI_ENCRYPTION_KEY"   # Pflicht
SECRET_BACKUPKEY  = "FREELLMAPI_DB_BACKUP_KEY"    # optional
SECRET_UNIFIEDKEY = "FREELLMAPI_UNIFIED_KEY"      # optional

# Dokumentierte Grenze aus package.json -> "engines": {"node": ">=20.18.0 <25.0.0"}
NODE_MIN = (20, 18, 0)
NODE_MAX = (25, 0, 0)

STATE = {
    "drive": False, "secret": False, "persist": False, "installed": False,
    "encryption_key": None, "backup_key": None, "unified_key": None,
    "proc": None, "restore_expected": False,
}

# ---------------------------- Ausgabe-Helfer --------------------------------
def ok(msg):    print("  [ OK ]  " + msg)
def info(msg):  print("  [ .. ]  " + msg)
def warn(msg):  print("  [WARN]  " + msg)

def fail(was, ursache, naechster):
    """Einheitliches Fehlerbild statt nackter Tracebacks."""
    print("")
    print("=" * 64)
    print("FEHLER:")
    print("  " + str(was))
    print("")
    print("MOEGLICHE URSACHE:")
    for line in str(ursache).splitlines():
        print("  " + line)
    print("")
    print("NAECHSTER SCHRITT:")
    for line in str(naechster).splitlines():
        print("  " + line)
    print("=" * 64)
    raise SystemExit(1)

# ---------------------------- Schwärzen von Secrets -------------------------
_RE_UNI    = re.compile(r"freellmapi-[0-9a-zA-Z]{12,}")
_RE_HEX64  = re.compile(r"\b[0-9a-fA-F]{64}\b")
_RE_SETUP  = re.compile(r"(First-run setup code:\s*)[A-Z0-9]{6,}")

def redact(text):
    """Entfernt Schluessel und Codes aus Text, BEVOR er ausgegeben wird."""
    text = _RE_UNI.sub("freellmapi-***REDACTED***", text)
    text = _RE_HEX64.sub("***REDACTED-64-HEX***", text)
    text = _RE_SETUP.sub(r"\1***REDACTED***", text)
    return text

def sh(cmd, cwd=None, timeout=300, env=None):
    """Kurzer Befehl, Ausgabe eingefangen."""
    p = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=str(cwd) if cwd else None,
                       capture_output=True, text=True, timeout=timeout, env=env)
    return p.returncode, p.stdout.strip(), p.stderr.strip()

def run_live(cmd, cwd=None, timeout=1800, env=None):
    """Langer Befehl mit Live-Ausgabe in die Zelle (npm, git, tar)."""
    t0 = time.time()
    rc = subprocess.call(cmd, shell=True, cwd=str(cwd) if cwd else None,
                         timeout=timeout, env=env)
    print("       (%.1f s)" % (time.time() - t0))
    return rc

def valid_encryption_key(v):
    """Exakt 64 Hex-Zeichen, nicht der .env.example-Platzhalter."""
    return bool(v) and v != "your-64-char-hex-key-here" and \
        re.fullmatch(r"[0-9a-fA-F]{64}", v) is not None

# ============================================================================
print("=" * 64)
print(" SCHRITT 1 — VORAUSSETZUNGEN")
print("=" * 64)

# ----------------------- 1) Colab-API verfügbar? ----------------------------
try:
    from google.colab import drive, userdata
except ImportError:
    fail("Das Modul 'google.colab' ist nicht verfuegbar.",
         "Diese Zelle laeuft nur in einer echten Google-Colab-Runtime, "
         "nicht in einem lokalen Jupyter.",
         "Das Notebook unter https://colab.research.google.com oeffnen.")

# ----------------------- 2) Google Drive mounten ----------------------------
if (DRIVE_MOUNT / "My Drive").is_dir():
    ok("Google Drive ist bereits gemountet")
else:
    try:
        drive.mount(str(DRIVE_MOUNT))
    except Exception as e:
        fail("Google Drive konnte nicht gemountet werden.",
             "%s: %s" % (type(e).__name__, e),
             "Im Browser die Google-Anmeldung erlauben und die Zelle erneut "
             "ausfuehren. Hilft das nicht: Laufzeit > Laufzeit neu starten.")

MYDRIVE = DRIVE_MOUNT / "My Drive"
if not MYDRIVE.is_dir():
    fail("'%s' existiert nach dem Mount nicht." % MYDRIVE,
         "Unvollstaendiger Mount oder ungewoehnliche Drive-Struktur.",
         "Laufzeit neu starten und Zelle 1 erneut ausfuehren.")
STATE["drive"] = True
ok("Google Drive gemountet: %s" % MYDRIVE)

# ----------------------- 3) Colab-Secrets laden -----------------------------
def read_secret(name, required):
    try:
        val = userdata.get(name)
    except userdata.SecretNotFoundError:
        if required:
            fail("Das Colab-Secret '%s' existiert nicht." % name,
                 "Der Secret wurde in diesem Colab-Konto noch nicht angelegt.",
                 "Linke Seitenleiste > Schluessel-Symbol (Secrets) >\n"
                 "'Neuen Secret hinzufuegen'\n"
                 "  Name : %s\n"
                 "  Wert : 64 Hex-Zeichen, erzeugt mit:  openssl rand -hex 32\n"
                 "  Schalter 'Notebook access' aktivieren." % name)
        return None
    except userdata.NotebookAccessError:
        if required:
            fail("Dieses Notebook hat keinen Zugriff auf das Secret '%s'." % name,
                 "Der Schalter 'Notebook access' ist fuer dieses Secret aus.",
                 "Secrets-Seitenleiste > beim Secret 'Notebook access' einschalten.")
        return None
    except userdata.TimeoutException:
        fail("Secret '%s' konnte nicht gelesen werden (Timeout)." % name,
             "Secrets funktionieren nur in der Colab-Weboberflaeche, nicht in "
             "exportierten Laeufen oder per Colab-CLI.",
             "Das Notebook in der Colab-Weboberflaeche ausfuehren.")
    val = (val or "").strip()
    if not val:
        if required:
            fail("Das Secret '%s' ist leer." % name,
                 "Der Secret wurde ohne Wert angelegt.",
                 "Secrets-Seitenleiste > Wert setzen (64 Hex-Zeichen).")
        return None
    return val

STATE["encryption_key"] = read_secret(SECRET_ENCRYPTION, required=True)
if not valid_encryption_key(STATE["encryption_key"]):
    fail("Das Secret '%s' hat kein gueltiges Format." % SECRET_ENCRYPTION,
         "FreeLLMAPI verlangt exakt 64 Hex-Zeichen (32-Byte-AES-256-GCM-Schluessel).\n"
         "Der Platzhalter 'your-64-char-hex-key-here' wird ausdruecklich abgelehnt\n"
         "(server/src/lib/crypto.ts).",
         "Neuen Schluessel erzeugen:  openssl rand -hex 32\n"
         "ACHTUNG: Sobald Provider-Keys gespeichert sind, darf dieser Schluessel\n"
         "nicht mehr geaendert werden - sonst ist alles Verschluesselte unlesbar.\n"
         "Ein Wechsel geht nur ueber 'npm run rotate-encryption-key' (docs/install.md).")
STATE["secret"] = True
ok("Secret '%s' geladen — 64 Hex-Zeichen, Wert wird nicht angezeigt" % SECRET_ENCRYPTION)

STATE["backup_key"] = read_secret(SECRET_BACKUPKEY, required=False)
if STATE["backup_key"] and not valid_encryption_key(STATE["backup_key"]):
    warn("'%s' ist kein gueltiges 64-Hex-Format -> wird ignoriert." % SECRET_BACKUPKEY)
    STATE["backup_key"] = None
if STATE["backup_key"]:
    ok("Secret '%s' geladen — getrennter Backup-Schluessel" % SECRET_BACKUPKEY)
else:
    info("Kein '%s' gesetzt -> FreeLLMAPI nutzt ENCRYPTION_KEY fuer die" % SECRET_BACKUPKEY)
    info("Backup-Huelle (dokumentierter Default in docs/env/01-variables.md).")

STATE["unified_key"] = read_secret(SECRET_UNIFIEDKEY, required=False)
if STATE["unified_key"]:
    ok("Secret '%s' geladen — Abschlusstest prueft /v1/models mit HTTP 200" % SECRET_UNIFIEDKEY)

# ----------------------- 4) Persistenz-Ordner in Drive ----------------------
try:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    probe = DRIVE_ROOT / ".schreibtest"
    probe.write_text("ok")
    probe.unlink()
except Exception as e:
    fail("Der Persistenz-Ordner %s ist nicht nutzbar." % DRIVE_ROOT,
         "%s: %s\nTypisch: Drive-Quota erschoepft, I/O-Fehler oder zu viele\n"
         "Dateien direkt im Drive-Hauptordner." % (type(e).__name__, e),
         "Drive-Quota pruefen, Papierkorb leeren, Dateien im Drive-Root in\n"
         "Unterordner verschieben. Danach Zelle 1 erneut ausfuehren.")
STATE["persist"] = True
ok("Persistenz-Ordner bereit: %s" % DRIVE_ROOT)

# ----------------------- 5) vorhandenen Datenbestand erkennen ---------------
def fsize(p):
    try:
        return p.stat().st_size
    except OSError:
        return -1

first_start = fsize(BACKUP_FILE) <= 0
print("")
info("Datenbestand in Google Drive:")
print("           Backup     : " + ("nicht vorhanden" if first_start
                                      else "vorhanden (%d Bytes)" % fsize(BACKUP_FILE)))
print("           Rollback   : " + ("nicht vorhanden" if fsize(BACKUP_PREV) <= 0
                                      else "%s (%d Bytes)" % (BACKUP_PREV.name, fsize(BACKUP_PREV))))
print("           App-Cache  : " + ("nicht vorhanden" if fsize(APP_CACHE) <= 0
                                      else "%s (%.1f MB)" % (APP_CACHE.name, fsize(APP_CACHE) / 1048576.0)))

print("")
print("  Drive:      " + ("OK" if STATE["drive"]    else "FEHLER"))
print("  Secret:     " + ("OK" if STATE["secret"]   else "FEHLER"))
print("  Persistenz: " + ("OK" if STATE["persist"]  else "FEHLER"))
print("")
if first_start:
    print("  Modus: ERSTER START — FreeLLMAPI wird frisch installiert,")
    print("         die Datenbank wird neu angelegt.")
else:
    print("  Modus: SPAETERER START — das verschluesselte Backup aus Drive wird")
    print("         wiederhergestellt, Provider-Keys sind danach wieder da.")


## Zelle 2 — Systemprüfung

Betriebssystem, Python, Node.js, npm, RAM, CPU, freier Speicher. Eine GPU wird **nicht**
benötigt. Docker wird geprüft, aber bewusst **nicht** verwendet (Begründung oben).

Node.js wird bei Bedarf direkt auf die von FreeLLMAPI geforderte Version gebracht
(`package.json` → `"engines": {"node": ">=20.18.0 <25.0.0"}``).

In [ ]:
# ============================================================================
#  ZELLE 2 — SYSTEMPRUEFUNG
# ============================================================================
print("=" * 64)
print(" SCHRITT 2 — SYSTEMPRUEFUNG")
print("=" * 64)

def ver_tuple(s):
    m = re.search(r"(\d+)\.(\d+)\.(\d+)", s or "")
    return tuple(int(x) for x in m.groups()) if m else (0, 0, 0)

def meminfo():
    total = avail = 0
    try:
        for line in Path("/proc/meminfo").read_text().splitlines():
            if line.startswith("MemTotal:"):
                total = int(line.split()[1])
            elif line.startswith("MemAvailable:"):
                avail = int(line.split()[1])
    except OSError:
        pass
    return total / 1048576.0, avail / 1048576.0   # GiB

# ---------------------------- Betriebssystem --------------------------------
ok("Betriebssystem : %s %s (%s)" % (platform.system(), platform.release(), platform.machine()))
ok("Python         : %s" % platform.python_version())

# ---------------------------- Node.js ---------------------------------------
rc, node_v, err = sh("node --version")
if rc != 0:
    fail("Node.js ist nicht installiert.",
         "FreeLLMAPI ist eine Node.js-Anwendung (server/package.json).",
         "Zelle 2 erneut ausfuehren - sie installiert Node.js 22 selbst, "
         "wenn es fehlt.")
    node_v = "0.0.0"
else:
    node_v = node_v.lstrip("v")

rc, npm_v, _ = sh("npm --version")
npm_v = npm_v if rc == 0 else "nicht gefunden"

need_node = not (NODE_MIN <= ver_tuple(node_v) < NODE_MAX)
ok("Node.js        : v%s%s" % (node_v, "   <-- ausserhalb des erlaubten Bereichs" if need_node else ""))
ok("npm            : %s" % npm_v)
info("Erforderlich laut package.json: >=20.18.0 <25.0.0")

if need_node:
    warn("Node.js wird auf Version 22 gebracht ...")
    if run_live("curl -fsSL https://deb.nodesource.com/setup_22.x -o /tmp/ns22.sh && bash /tmp/ns22.sh") != 0:
        fail("Das NodeSource-Setup-Skript konnte nicht ausgefuehrt werden.",
             "Kein Zugriff auf deb.nodesource.com oder apt ist blockiert.",
             "Zelle erneut ausfuehren; ggf. 'Laufzeit > Laufzeit neu starten'.")
    if run_live("apt-get install -y nodejs") != 0:
        fail("'apt-get install nodejs' ist fehlgeschlagen.",
             "Das NodeSource-Repository wurde nicht korrekt eingerichtet.",
             "Zelle erneut ausfuehren und die Ausgabe oben pruefen.")
    rc, node_v, _ = sh("node --version")
    node_v = node_v.lstrip("v")
    rc, npm_v, _ = sh("npm --version")
    if not (NODE_MIN <= ver_tuple(node_v) < NODE_MAX):
        fail("Node.js ist nach der Installation immer noch v%s." % node_v,
             "Die Runtime liefert eine andere Version als erwartet.",
             "FREELLMAPI_REF ggf. auf einen aelteren Tag setzen oder die "
             "Node-Installation manuell pruefen:  which node; node --version")
    ok("Node.js jetzt  : v%s (npm %s)" % (node_v, npm_v))

# ---------------------------- Ressourcen ------------------------------------
total, avail = meminfo()
ok("RAM            : %.1f GiB gesamt, %.1f GiB verfuegbar" % (total, avail))
ok("CPU-Kerne      : %s" % os.cpu_count())
du = shutil.disk_usage("/content")
ok("Freier Speicher: %.1f GiB von %.1f GiB auf /content" % (du.free / 1073741824.0,
                                                            du.total / 1073741824.0))
if avail < 1.5:
    warn("Wenig freier RAM. FreeLLMAPI laeuft, aber der Build kann knapp werden.")
if du.free < 3 * 1073741824:
    fail("Zu wenig freier Speicher auf /content.",
         "node_modules braucht rund 500 MB, dazu der Build.",
         "Laufzeit neu starten (setzt /content zurueck) und erneut versuchen.")

# ---------------------------- GPU (nur informativ) --------------------------
rc, gpu, _ = sh("nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null")
if rc == 0 and gpu:
    info("GPU gefunden   : %s" % gpu.replace("\n", ", "))
    info("               Fuer FreeLLMAPI nicht noetig - es wird nichts lokal inferiert.")
else:
    info("GPU            : keine (nicht erforderlich)")

# ---------------------------- Docker (bewusst ungenutzt) --------------------
rc, dock, _ = sh("docker --version 2>/dev/null")
if rc == 0 and dock:
    info("Docker         : %s vorhanden, wird aber NICHT verwendet" % dock)
else:
    info("Docker         : nicht verfuegbar")
info("               FreeLLMAPI empfiehlt Docker, auf verwalteten Colab-Runtimes")
info("               gibt es aber keinen Docker-Daemon, und die Colab-FAQ untersagt")
info("               Container-Techniken zur Umgehung der Anti-Missbrauchs-Regeln.")
info("               Darum: der dokumentierte Weg ohne Docker (docs/install.md).")

# ---------------------------- Build-Werkzeuge -------------------------------
missing = [t for t in ("python3", "make", "g++") if sh("%s --version" % t)[0] != 0]
if missing:
    warn("Build-Werkzeuge fehlen: %s -> werden installiert" % ", ".join(missing))
    if run_live("apt-get update -qq && apt-get install -y python3 make g++") != 0:
        warn("Installation der Build-Werkzeuge fehlgeschlagen. Nur noetig, falls "
             "better-sqlite3 nicht als Fertigbuild ladbar ist.")
else:
    ok("Build-Werkzeuge: python3, make, g++ vorhanden (Fallback fuer better-sqlite3)")

print("")
print("  Systempruefung abgeschlossen.")


## Zelle 3 — Installation

Reproduzierbar auf den Tag `v0.9.5` gepinnt. Drei Wege, in dieser Reihenfolge:

1. **Bereits installiert** (gleiche Laufzeit, Zelle erneut ausgeführt) → weiter
2. **Build-Cache in Drive** vorhanden → Tarball nach `/content` kopieren und auspacken
3. **Neu installieren** → `git clone` → `npm ci` → `npm run build` → Cache nach Drive

> **Bekanntes Problem im Upstream-Repo:** `package-lock.json` enthält (Stand `v0.9.5`) 40
> Einträge, die auf `registry.npmmirror.com` zeigen. Ist dieser Mirror nicht erreichbar,
> bricht `npm ci` mit `Exit handler never called!` ab. Die Zelle schreibt den Host auf
> `registry.npmjs.org` um — gleicher Pfad, gleiche Integrity-Hashes, nur ein anderer Host.

> **`better-sqlite3` ist Pflicht:** Es ist nur eine *optionale* Abhängigkeit, npm schluckt
> einen Fehlschlag also still — der Server würde dann mit
> `better-sqlite3 is not installed` abbrechen. Der `node:sqlite`-Fallback greift laut
> `server/src/db/index.ts` nur bei `process.platform === 'android'`. Deshalb wird das
> Paket hier explizit installiert **und verifiziert**.

In [ ]:
# ============================================================================
#  ZELLE 3 — INSTALLATION
# ============================================================================
print("=" * 64)
print(" SCHRITT 3 — INSTALLATION (FreeLLMAPI %s)" % FREELLMAPI_REF)
print("=" * 64)

def build_present():
    """Server- und Dashboard-Build vorhanden?"""
    return (APP_DIR / "server" / "dist" / "index.js").is_file() \
        and (APP_DIR / "client" / "dist" / "index.html").is_file()

def ref_matches():
    """Stammt die Installation aus dem in Zelle 1 gepinnten Tag?"""
    return REF_MARK.is_file() and REF_MARK.read_text().strip() == FREELLMAPI_REF

def app_ready():
    return build_present() and ref_matches()

def missing_build_parts():
    m = []
    if not (APP_DIR / "server" / "dist" / "index.js").is_file():
        m.append("server/dist/index.js")
    if not (APP_DIR / "client" / "dist" / "index.html").is_file():
        m.append("client/dist/index.html")
    return m

def bs3_ok():
    """better-sqlite3 muss auf Linux zwingend ladbar sein."""
    if not APP_DIR.is_dir():
        return False
    return sh("node -e \"require('better-sqlite3')\"", cwd=APP_DIR, timeout=60)[0] == 0

def bs3_pinned_version():
    try:
        lock = json.loads((APP_DIR / "package-lock.json").read_text())
        return lock["packages"]["node_modules/better-sqlite3"]["version"]
    except Exception:
        return "latest"

def install_better_sqlite3():
    """Explizit installieren - als optionale Abhaengigkeit wuerde npm einen
    Fehlschlag still ueberspringen."""
    ver = bs3_pinned_version()
    info("better-sqlite3@%s wird installiert (Fertigbinary, sonst node-gyp) ..." % ver)
    run_live("npm install --no-save --no-audit --no-fund better-sqlite3@%s" % ver,
             cwd=APP_DIR, timeout=1800)
    if not bs3_ok():
        warn("Fertigbinary nicht ladbar -> Build aus den Quellen ...")
        run_live("apt-get install -y python3 make g++ >/dev/null 2>&1 || true")
        run_live("npm install --no-save --no-audit --no-fund --build-from-source "
                 "better-sqlite3@%s" % ver, cwd=APP_DIR, timeout=1800)
    return bs3_ok()

def fix_npmmirror():
    """40 Eintraege im Lockfile zeigen auf registry.npmmirror.com. Gleiche Pfade
    und Hashes auf registry.npmjs.org, nur der Host ist ein anderer."""
    lock = APP_DIR / "package-lock.json"
    raw = lock.read_text()
    n = raw.count("registry.npmmirror.com")
    if n:
        lock.write_text(raw.replace("https://registry.npmmirror.com/",
                                    "https://registry.npmjs.org/"))
        warn("%d Lockfile-Eintraege von registry.npmmirror.com auf" % n)
        warn("registry.npmjs.org umgeschrieben (Versionen und Hashes unveraendert).")
    return n

def check_engines():
    """Die echte Grenze aus dem geklonten package.json lesen, nicht raten."""
    try:
        pkg = json.loads((APP_DIR / "package.json").read_text())
        want = pkg.get("engines", {}).get("node", "")
        info("package.json -> engines.node = %s" % (want or "(nicht angegeben)"))
        v = ver_tuple(sh("node --version")[1].lstrip("v"))
        if not (NODE_MIN <= v < NODE_MAX):
            fail("Node.js v%s erfuellt die dokumentierte Grenze %s nicht."
                 % (".".join(map(str, v)), want),
                 "Die Runtime liefert eine unpassende Node-Version.",
                 "Zelle 2 erneut ausfuehren - sie installiert Node.js 22.")
    except FileNotFoundError:
        pass

def make_cache():
    """Nur Code + node_modules + dist, niemals Daten oder Schluessel."""
    if fsize(APP_CACHE) > 0:
        info("App-Cache in Drive vorhanden, wird nicht neu geschrieben.")
        return
    info("App-Cache fuer naechste Starts wird nach Drive geschrieben (~110 MB) ...")
    rc = run_live("tar czf %s --exclude=./.git --exclude=./data --exclude=./logs "
                  "--exclude=./server/data --exclude=./.env.example ."
                  % shlex.quote(str(APP_CACHE)), cwd=APP_DIR, timeout=900)
    if rc != 0 or fsize(APP_CACHE) <= 0:
        warn("App-Cache konnte nicht geschrieben werden. Kein Problem - der "
             "naechste Start installiert dann einfach neu.")
    else:
        ok("App-Cache gespeichert: %s (%.1f MB)" % (APP_CACHE.name, fsize(APP_CACHE) / 1048576.0))

def use_cache():
    info("Build-Cache aus Google Drive wird entpackt ...")
    rc = run_live("tar xzf %s -C %s" % (shlex.quote(str(APP_CACHE)),
                                        shlex.quote(str(APP_DIR))),
                  cwd="/content", timeout=900)
    return rc == 0

# ------------------------------ Entscheidung --------------------------------
if app_ready() and bs3_ok():
    ok("FreeLLMAPI %s ist bereits installiert -> weiter" % FREELLMAPI_REF)
else:
    # 2) Build-Cache aus Drive?
    if not build_present() and fsize(APP_CACHE) > 0:
        APP_DIR.mkdir(parents=True, exist_ok=True)
        if use_cache() and build_present():
            ok("FreeLLMAPI %s aus dem Drive-Cache wiederhergestellt" % FREELLMAPI_REF)
            REF_MARK.write_text(FREELLMAPI_REF)
        else:
            warn("Cache unbrauchbar -> Neuinstallation")
            shutil.rmtree(APP_DIR, ignore_errors=True)
            try:
                APP_CACHE.unlink()
            except OSError:
                pass

    # 3) Neu installieren
    if not build_present():
        info("Neuinstallation von FreeLLMAPI %s — das dauert 3-6 Minuten." % FREELLMAPI_REF)
        shutil.rmtree(APP_DIR, ignore_errors=True)

        rc = run_live("git clone --depth 1 --branch %s %s %s"
                      % (shlex.quote(FREELLMAPI_REF), shlex.quote(REPO_URL),
                         shlex.quote(str(APP_DIR))), timeout=900)
        if rc != 0 or not (APP_DIR / "package.json").is_file():
            fail("git clone ist fehlgeschlagen.",
                 "Kein Zugriff auf github.com, oder der Tag '%s' existiert nicht."
                 % FREELLMAPI_REF,
                 "https://github.com/tashfeenahmed/freellmapi/tags oeffnen und "
                 "FREELLMAPI_REF in Zelle 1 auf einen vorhandenen Tag setzen.")
        ok("Repository geklont (Tag %s)" % FREELLMAPI_REF)

        fix_npmmirror()
        check_engines()

        info("npm ci — Abhaengigkeiten werden installiert ...")
        if run_live("npm ci --no-audit --no-fund --loglevel=error", cwd=APP_DIR,
                    timeout=1800) != 0:
            fail("'npm ci' ist fehlgeschlagen.",
                 "Netzproblem beim npm-Registry-Zugriff, oder das Lockfile passt "
                 "nicht zur npm-Version.",
                 "Zelle 3 erneut ausfuehren. Bei wiederholtem Fehler: Ausgabe oben "
                 "pruefen und ggf. 'Laufzeit neu starten'.")
        ok("Abhaengigkeiten installiert")

        info("npm run build — Server, CLI und Dashboard werden gebaut (~30 s) ...")
        if run_live("npm run build --loglevel=error", cwd=APP_DIR, timeout=1800) != 0:
            fail("'npm run build' ist fehlgeschlagen.",
                 "TypeScript- oder Vite-Buildfehler, meist durch zu wenig RAM.",
                 "Laufzeit neu starten und Zelle 3 erneut ausfuehren.")
        if not build_present():
            fail("Der Build lief durch, aber es fehlt: %s."
                 % ", ".join(missing_build_parts()),
                 "Der Build meldete keinen Fehler, hat aber nicht alle Artefakte "
                 "erzeugt.",
                 "Ausgabe oben pruefen; ggf. FREELLMAPI_REF in Zelle 1 auf 'main' "
                 "setzen und Zelle 3 erneut ausfuehren.")
        REF_MARK.write_text(FREELLMAPI_REF)
        ok("Build abgeschlossen (server/dist + client/dist)")

    # better-sqlite3 IMMER am Ende pruefen - als optionale Abhaengigkeit
    # ueberspringt npm einen Fehlschlag sonst still.
    if bs3_ok():
        ok("better-sqlite3 ladbar")
    elif install_better_sqlite3():
        ok("better-sqlite3 installiert")
    else:
        fail("better-sqlite3 laesst sich nicht installieren.",
             "Weder Fertigbinary noch Quellbuild waren moeglich. Auf Linux ist "
             "das Modul zwingend noetig (server/src/db/index.ts).",
             "Zelle 2 erneut ausfuehren (installiert python3/make/g++), APP_CACHE "
             "in Drive loeschen und Zelle 3 wiederholen.")

REF_MARK.write_text(FREELLMAPI_REF)
STATE["installed"] = True
make_cache()

print("")
print("  Installation : OK")
print("  Server       : %s" % (APP_DIR / "server" / "dist" / "index.js"))
print("  Dashboard    : %s" % (APP_DIR / "client" / "dist" / "index.html"))
print("  better-sqlite3: ladbar")


## Zelle 4 — Persistente Datenbank

**Warum die Datenbank nicht direkt in Google Drive liegt:** FreeLLMAPI öffnet SQLite im
WAL-Modus (`journal_mode = WAL` in `server/src/db/index.ts`), erzeugt also `-wal`/`-shm`-
Sidecars und braucht funktionierende Datei-Locks. Auf einem FUSE-Mount wie Google Drive ist
das unzuverlässig — die Colab-FAQ warnt ausdrücklich vor I/O-Fehlern und Datenverlust bei
unterbrochenen Drive-Operationen.

Stattdessen nutzt dieses Notebook exakt das, was FreeLLMAPI selbst für Hosts mit flüchtigen
Disks dokumentiert (`docs/deployment/02-updates-and-backup.md`):

```
SQLite lokal in /content      ←  schnell, WAL-sicher
        │
        ├── Start:  FREEAPI_DB_BACKUP_PATH wird automatisch zurueckgespielt,
        │           wenn die DB-Datei fehlt ("before migrations run")
        └── laufend: verschluesseltes Backup alle FREEAPI_DB_BACKUP_INTERVAL_MS
```

**Zusätzlicher Schutz durch dieses Notebook.** FreeLLMAPI überspringt den Restore, sobald die
Zieldatei existiert und größer als 0 Byte ist (`restoreDbBackupIfNeeded` in
`server/src/lib/db-backup.ts`). Ein Fehlstart *nach* dem Anlegen der DB hinterlässt aber eine
leere 4-kB-Datenbank — der nächste Start würde dann mit leerer DB hochkommen und die
Backup-Pumpe das gute Drive-Backup überschreiben. Das wurde hier reproduziert. Dagegen:

1. **Preflight** — der `ENCRYPTION_KEY` wird *vor* dem Start geprüft (die beiden häufigsten
   Fehlstarts passieren genau dort).
2. **Rollback-Kopie** — vor dem Start wird das gültige Drive-Backup nach `.prev` kopiert.
3. **Aufräumen** — Zelle 5 löscht eine von einem Fehlstart halb angelegte DB wieder.

In [ ]:
# ============================================================================
#  ZELLE 4 — PERSISTENTE DATENBANK + BACKUP-SCHUTZ
# ============================================================================
print("=" * 64)
print(" SCHRITT 4 — PERSISTENTE DATENBANK")
print("=" * 64)

# Kopfbalken des verschluesselten Backups, server/src/lib/db-backup.ts:
#   MAGIC 'FAPIBK1\0' + IV(12) + GCM-Tag(16) + AES-256-GCM(gzip(sqlite))
BACKUP_MAGIC = b"FAPIBK1\x00"

def backup_status(p):
    n = fsize(p)
    if n <= 0:
        return "fehlt", False
    if n < len(BACKUP_MAGIC) + 12 + 16:
        return "ungueltig (nur %d Bytes)" % n, False
    try:
        with open(p, "rb") as f:
            head = f.read(len(BACKUP_MAGIC))
    except OSError as e:
        return "nicht lesbar (%s)" % e.__class__.__name__, False
    if head != BACKUP_MAGIC:
        return "ungueltig (kein FAPIBK1-Header)", False
    return "gueltig (%d Bytes)" % n, True

# ---------------------------- Preflight Schluessel --------------------------
# BEVOR der Server startet: die beiden haeufigsten Fehlstarts passieren genau
# hier, und ein Fehlstart nach initDb() hinterlaesst eine leere Datenbank, die
# den Restore beim naechsten Start blockiert.
if not valid_encryption_key(STATE["encryption_key"]):
    fail("Preflight: ENCRYPTION_KEY ist ungueltig.",
         "FreeLLMAPI braucht exakt 64 Hex-Zeichen (server/src/lib/crypto.ts).",
         "Secret '%s' in Colab korrigieren und Zelle 1 erneut ausfuehren."
         % SECRET_ENCRYPTION)
ok("Preflight ENCRYPTION_KEY : 64 Hex-Zeichen, kein Platzhalter")
if STATE["backup_key"]:
    ok("Preflight BACKUP_KEY   : 64 Hex-Zeichen, getrennter Schluessel")
else:
    info("Preflight BACKUP_KEY   : nicht gesetzt -> ENCRYPTION_KEY (Default)")

# ---------------------------- Zustand der Backups ---------------------------
st_main, good_main = backup_status(BACKUP_FILE)
st_prev, good_prev = backup_status(BACKUP_PREV)
print("")
info("Haupt-Backup  %s : %s" % (BACKUP_FILE.name, st_main))
info("Rollback      %s : %s" % (BACKUP_PREV.name, st_prev))

STATE["restore_expected"] = False

if good_main:
    # Bekannten guten Stand wegsichern, BEVOR diese Sitzung das Backup
    # ueberschreiben kann (die Backup-Pumpe laeuft sofort nach dem Start).
    try:
        shutil.copy2(BACKUP_FILE, BACKUP_PREV)
        ok("Rollback-Kopie aktualisiert: %s" % BACKUP_PREV.name)
    except OSError as e:
        warn("Rollback-Kopie fehlgeschlagen (%s) - es wird ohne sie gestartet."
             % e.__class__.__name__)
    if fsize(DB_PATH) > 0:
        info("Lokale Datenbank vorhanden (dieselbe Laufzeit) -> kein Restore noetig.")
    else:
        STATE["restore_expected"] = True
        info("Lokale Datenbank fehlt -> FreeLLMAPI spielt das Backup beim Start ein.")

elif good_prev and fsize(DB_PATH) <= 0:
    shutil.copy2(BACKUP_PREV, BACKUP_FILE)
    warn("Haupt-Backup unbrauchbar -> Rollback-Kopie wurde zum Haupt-Backup.")
    STATE["restore_expected"] = True

elif fsize(DB_PATH) > 0:
    info("Lokale Datenbank vorhanden, kein Backup in Drive -> sie wird benutzt.")

else:
    info("Kein nutzbares Backup und keine lokale Datenbank")
    info("-> FreeLLMAPI legt beim Start eine NEUE Datenbank an (ERSTER START).")

# ---------------------------- Lokales Datenverzeichnis ----------------------
try:
    DB_DIR.mkdir(parents=True, exist_ok=True)
    os.chmod(DB_DIR, 0o700)
except OSError as e:
    fail("Das lokale Datenverzeichnis %s konnte nicht angelegt werden." % DB_DIR,
         "%s: %s" % (type(e).__name__, e),
         "Zelle erneut ausfuehren; ggf. Laufzeit neu starten.")
ok("Lokales Datenverzeichnis: %s (Modus 0700)" % DB_DIR)
ok("Datenbankpfad (FREEAPI_DB_PATH): %s" % DB_PATH)

print("")
print("  Persistente Daten in Google Drive:")
print("    %s   <- verschluesselte SQLite (AES-256-GCM + gzip)" % BACKUP_FILE.name)
print("    %s   <- Rollback-Kopie, ebenfalls verschluesselt" % BACKUP_PREV.name)
print("    %s   <- nur Code, keine Schluessel, keine Provider-Keys" % APP_CACHE.name)
print("")
print("  NICHT in Drive: ENCRYPTION_KEY, Provider-Keys im Klartext, .env, Server-Log")
print("")
print("  Datenbank     : OK")


## Zelle 5 — FreeLLMAPI starten

Start mit dem dokumentierten Befehl für den Betrieb ohne Docker:

```
node server/dist/index.js          # Server + Dashboard, beides auf :3001
```

Alle Einstellungen werden als **Umgebungsvariablen an den Prozess** übergeben. Es wird
bewusst **keine `.env`-Datei** geschrieben — weder in `/content` noch in Google Drive.
`FREEAPI_ENV_PATH` zeigt auf eine nicht vorhandene Datei, damit `dotenv` nichts lädt
(dokumentiert: „dotenv silently no-ops on a missing file").

In [ ]:
# ============================================================================
#  ZELLE 5 — FREE LLM API STARTEN
# ============================================================================
print("=" * 64)
print(" SCHRITT 5 — FREE LLM API STARTEN")
print("=" * 64)

def http_get(path, timeout=8, headers=None):
    url = "http://127.0.0.1:%d%s" % (PORT, path)
    req = urllib.request.Request(url, headers=headers or {})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, r.read()
    except urllib.error.HTTPError as e:
        return e.code, e.read()
    except Exception as e:
        return None, "%s: %s" % (type(e).__name__, e)

def stop_server(verbose=True):
    stopped = False
    if PID_FILE.exists():
        try:
            os.kill(int(PID_FILE.read_text().strip()), signal.SIGTERM)
            stopped = True
        except (ValueError, ProcessLookupError, PermissionError):
            pass
        PID_FILE.unlink(missing_ok=True)
    p = STATE.get("proc")
    if p is not None and p.poll() is None:
        try:
            p.terminate()
            stopped = True
        except Exception:
            pass
    if stopped:
        time.sleep(2)
    sh("pkill -f 'node .*server/dist/index.js' 2>/dev/null || true")
    time.sleep(1)
    if verbose and stopped:
        ok("FreeLLMAPI gestoppt")
    return stopped

def log_text(n=4000):
    try:
        return redact(LOG_FILE.read_text(errors="replace")[-n:])
    except OSError:
        return "(kein Log gefunden)"

def diagnose(tail):
    """Bekannte Fehlerbilder aus server/src auf Ursache + naechsten Schritt mappen."""
    if "could not restore SQLite backup" in tail:
        return ("Das verschluesselte Backup in Google Drive konnte nicht entschluesselt "
                "werden. Fast immer passt ENCRYPTION_KEY nicht zum Backup.",
                "Den urspruenglichen ENCRYPTION_KEY wieder als Colab-Secret setzen.\n"
                "Ist er wirklich verloren, sind die gespeicherten Provider-Keys nicht\n"
                "mehr rettbar (docs/env/02-security-and-keys.md). Dann das Backup\n"
                "beiseitelegen:  mv '%s' '%s.alt'  und neu starten."
                % (BACKUP_FILE.name, BACKUP_FILE.name))
    if "ENCRYPTION_KEY is required in production" in tail:
        return ("Der Server lief ohne ENCRYPTION_KEY.",
                "Zelle 1 erneut ausfuehren und pruefen, ob das Secret geladen wird.")
    if "Invalid ENCRYPTION_KEY" in tail:
        return ("ENCRYPTION_KEY hat nicht exakt 64 Hex-Zeichen.",
                "openssl rand -hex 32  ausfuehren und den Wert als Secret setzen.")
    if "better-sqlite3 is not installed" in tail:
        return ("Das native SQLite-Modul fehlt.",
                "Zelle 3 erneut ausfuehren - sie installiert better-sqlite3 explizit.")
    if "EADDRINUSE" in tail:
        return ("Port %d ist bereits belegt." % PORT,
                "Zelle 5 erneut ausfuehren (sie stoppt einen Altprozess), oder in\n"
                "Zelle 1 einen anderen PORT waehlen.")
    if "Cannot find module" in tail or "Cannot find package" in tail:
        return ("Der Build oder node_modules ist unvollstaendig.",
                "APP_CACHE in Drive loeschen und Zelle 3 erneut ausfuehren.")
    if "Failed to start" in tail:
        return ("Der Server hat den Start abgebrochen (siehe Log unten).",
                "Log lesen; bei Datenbankfehlern hilft Zelle 4 erneut ausfuehren.")
    return ("Der Prozess hat den Health-Endpunkt nicht rechtzeitig erreicht.",
            "Zelle 5 erneut ausfuehren. Wiederholt es sich, das Log unten pruefen.")

# --------------------- Port bereits belegt? ---------------------------------
st, _ = http_get("/api/ping", timeout=3)
if st == 200:
    warn("Auf Port %d laeuft bereits ein FreeLLMAPI-Prozess -> wird gestoppt." % PORT)
    stop_server(verbose=False)
    st, _ = http_get("/api/ping", timeout=3)
    if st == 200:
        fail("Der Altprozess auf Port %d laesst sich nicht beenden." % PORT,
             "Der Prozess gehoert vermutlich zu einer anderen Sitzung und "
             "reagiert nicht auf SIGTERM.",
             "In Zelle 1 einen anderen PORT setzen, oder die Laufzeit neu "
             "starten (beendet alle Prozesse).")
    ok("Altprozess beendet, Port %d ist frei" % PORT)

# --------------------- Umgebung fuer den Prozess ----------------------------
env = os.environ.copy()
env.update({
    "NODE_ENV": "production",                       # macht ENCRYPTION_KEY zur Pflicht
    "PORT": str(PORT),                              # docs/env/01-variables.md
    "HOST": HOST,                                   # nur loopback
    "FREEAPI_DB_PATH": str(DB_PATH),                # SQLite lokal (WAL-sicher)
    "FREEAPI_DB_DIR_HARDENING": "1",                # Verzeichnis 0700 erzwingen
    "FREEAPI_DB_BACKUP_PATH": str(BACKUP_FILE),     # verschluesselt nach Drive
    "FREEAPI_DB_BACKUP_INTERVAL_MS": str(BACKUP_INTERVAL_MS),
    "FREEAPI_ENV_PATH": str(APP_DIR / ".env.colab-absent"),  # keine .env laden
    "FREELLMAPI_UPDATE_CHECK": "off",               # kein Update-Check-Request
    "ENCRYPTION_KEY": STATE["encryption_key"],      # nur im Prozess-Speicher
})
if STATE["backup_key"]:
    env["FREEAPI_DB_BACKUP_KEY"] = STATE["backup_key"]

# --------------------- Start -------------------------------------------------
DB_EXISTED_BEFORE = fsize(DB_PATH) > 0
LOG_DIR.mkdir(parents=True, exist_ok=True)
try:
    LOG_FILE.write_text("")            # Log dieser Sitzung (bleibt in /content)
    _log = open(LOG_FILE, "ab", buffering=0)
    proc = subprocess.Popen(["node", "server/dist/index.js"], cwd=str(APP_DIR),
                            env=env, stdout=_log, stderr=subprocess.STDOUT,
                            stdin=subprocess.DEVNULL, start_new_session=True)
except Exception as e:
    fail("FreeLLMAPI konnte nicht gestartet werden.",
         "%s: %s" % (type(e).__name__, e),
         "Zelle 3 erneut ausfuehren (Installation pruefen).")
STATE["proc"] = proc
PID_FILE.write_text(str(proc.pid))
info("Prozess gestartet (PID %d), warte auf /api/ping ..." % proc.pid)

started = False
deadline = time.time() + 90
while time.time() < deadline:
    if proc.poll() is not None:
        break
    st, _ = http_get("/api/ping", timeout=3)
    if st == 200:
        started = True
        break
    time.sleep(1)

if not started:
    # Halb angelegte, leere Datenbank entfernen: sonst blockiert sie beim
    # naechsten Start den Restore und die Backup-Pumpe ueberschreibt das
    # gute Drive-Backup.
    if not DB_EXISTED_BEFORE and fsize(DB_PATH) > 0:
        for suffix in ("", "-wal", "-shm"):
            Path(str(DB_PATH) + suffix).unlink(missing_ok=True)
        warn("Halb angelegte leere Datenbank entfernt - der naechste Start kann "
             "das Drive-Backup wiederherstellen.")
    tail = log_text()
    ursache, schritt = diagnose(tail)
    fail("FreeLLMAPI ist nicht gestartet (Exit-Code %s)." % proc.poll(),
         ursache + "\n\n--- Server-Log, Schluessel geschwaerzt ---\n" + tail,
         schritt)

# --------------------- Erfolg vermelden -------------------------------------
ok("FreeLLMAPI laeuft (PID %d)" % proc.pid)
ok("Health-Endpunkt GET /api/ping antwortet mit HTTP 200")
ok("Gebunden auf %s:%d — nur innerhalb der Colab-Runtime erreichbar" % (HOST, PORT))

tail = log_text()
if "[db-backup] restored" in tail:
    m = re.search(r"\[db-backup\] restored (\d+) bytes", tail)
    ok("Verschluesseltes Backup aus Drive wiederhergestellt (%s Bytes)"
       % (m.group(1) if m else "?"))
elif "Seeded" in tail:
    info("Neue Datenbank angelegt (ERSTER START) — Provider-Keys jetzt im "
         "Dashboard unter 'Keys' eintragen.")
if "[db-backup] uploaded" in tail:
    ok("Erstes verschluesseltes Backup nach Drive geschrieben")
if "could not restrict permissions" in tail:
    info("Hinweis: Drive kann die Dateirechte 0600 nicht halten. Die Datei ist "
         "trotzdem verschluesselt und nur fuer dein Google-Konto sichtbar.")

print("")
print("  Server        : OK  (PID %d)" % proc.pid)
print("  Port          : OK  (%d, loopback)" % PORT)
print("  API           : OK  (/api/ping = 200)")
print("")
print("  Server-Log (lokal, nicht in Drive): %s" % LOG_FILE)


## Zelle 6 — Health-Check und Dashboard

Geprüft werden Server, API und der Model-Endpunkt. Danach wird das Dashboard über die
offizielle Colab-Funktion `google.colab.output.serve_kernel_port_as_iframe()` eingebettet.

> `serve_kernel_port_as_window()` ist laut colabtools-Quellcode **deprecated** („Browser
> security updates have broken this feature") — deshalb das iFrame.

In [ ]:
# ============================================================================
#  ZELLE 6 — HEALTH-CHECK + DASHBOARD
# ============================================================================
print("=" * 64)
print(" SCHRITT 6 — HEALTH-CHECK")
print("=" * 64)

def row(label, good, detail=""):
    print("  %-14s %s%s" % (label + ":", "OK  " if good else "FEHLER",
                            ("  — " + detail) if detail else ""))
    return good

res = {}
p = STATE.get("proc")
res["runtime"] = row("Runtime", p is not None and p.poll() is None,
                     "PID %s" % (p.pid if p else "-"))

wtest = DRIVE_ROOT / ".schreibtest2"
try:
    wtest.write_text("ok"); wtest.unlink(); res["drive"] = row("Drive", True, str(DRIVE_ROOT))
except OSError as e:
    res["drive"] = row("Drive", False, e.__class__.__name__)

res["secret"] = row("Secret", bool(STATE["secret"]),
                    "%s geladen" % SECRET_ENCRYPTION)
res["install"] = row("Installation", app_ready(), "FreeLLMAPI %s" % FREELLMAPI_REF)
res["db"] = row("Database", fsize(DB_PATH) > 0,
                "%s (%d Bytes)" % (DB_PATH.name, fsize(DB_PATH)))
st_b, good_b = backup_status(BACKUP_FILE)
res["backup"] = row("Backup", good_b, st_b)

st, _ = http_get("/api/ping", timeout=8)
res["server"] = row("Server", st == 200, "GET /api/ping -> %s" % st)

st, body = http_get("/v1/openapi.json", timeout=10)
res["api"] = row("API", st == 200, "GET /v1/openapi.json -> %s" % st)

if STATE["unified_key"]:
    st, body = http_get("/v1/models", timeout=20,
                        headers={"Authorization": "Bearer " + STATE["unified_key"]})
    n = ""
    if st == 200:
        try:
            n = "%d Modelle" % len(json.loads(body).get("data", []))
        except Exception:
            n = ""
    res["models"] = row("Models", st == 200, "GET /v1/models -> %s %s" % (st, n))
else:
    st, _ = http_get("/v1/models", timeout=15)
    res["models"] = row("Models", st == 401,
                        "GET /v1/models -> %s (ohne Unified Key erwartet: 401)" % st)

print("")
if all(res.values()):
    print("  FreeLLMAPI laeuft.")
else:
    print("  Nicht alle Pruefungen erfolgreich — Details oben.")

# ---------------------------- Dashboard -------------------------------------
print("")
print("=" * 64)
print(" DASHBOARD")
print("=" * 64)
try:
    from google.colab import output as colab_output
    try:
        url = colab_output.eval_js("google.colab.kernel.proxyPort(%d)" % PORT,
                                   timeout_sec=15)
        if isinstance(url, str) and url.startswith("http"):
            ok("Dashboard-URL: %s" % url.rstrip("/") + "/")
    except Exception:
        pass
    info("Das Dashboard wird unten als iFrame eingeblendet (Hoehe 800 px).")
    colab_output.serve_kernel_port_as_iframe(PORT, path="/", width="100%", height="800")
except ImportError:
    warn("google.colab.output nicht verfuegbar - Dashboard manuell oeffnen.")

print("")
print("  OEFFENTLICH NICHT ERREICHBAR. Kein Tunnel, kein Proxy (Version 1).")
print("")
print("  ERSTER START:")
print("    1. Im Dashboard einen Account anlegen (E-Mail + Passwort).")
print("       Die Anfrage kommt ueber loopback, deshalb ist kein Setup-Code noetig.")
print("    2. Seite 'Keys': Provider-Keys eintragen (werden AES-256-GCM-verschluesselt).")
print("    3. Seite 'Keys', Kopfzeile: den Unified Key 'freellmapi-...' kopieren.")
print("       -> Base-URL fuer OpenAI-Clients: http://127.0.0.1:%d/v1" % PORT)
print("    4. Optional: den Unified Key als Colab-Secret '%s' ablegen," % SECRET_UNIFIEDKEY)
print("       dann prueft Zelle 7 /v1/models mit HTTP 200.")
print("")
print("  SPAETERER START: Zellen 1-7 ausfuehren. Account, Provider-Keys und")
print("  Fallback-Ketten kommen verschluesselt aus Google Drive zurueck.")


## Zelle 7 — Abschlusstest

Neun Prüfungen, darunter eine, die aktiv sucht, ob der `ENCRYPTION_KEY` irgendwo in
Google Drive gelandet ist.

In [ ]:
# ============================================================================
#  ZELLE 7 — ABSCHLUSSTEST
# ============================================================================
print("=" * 64)
print(" SCHRITT 7 — ABSCHLUSSTEST")
print("=" * 64)

def check(label, condition, note=""):
    tag = "[OK]     " if condition else "[FEHLER] "
    print("  %s %-24s %s" % (tag, label, note))
    return bool(condition)

def skip(label, why):
    print("  [UEBERSPR.] %-24s %s" % (label, why))
    return None

results = []
p = STATE.get("proc")

results.append(check("Colab Runtime",
                     p is not None and p.poll() is None,
                     "Node-Prozess laeuft, PID %s" % (p.pid if p else "-")))
results.append(check("Google Drive",
                     (DRIVE_MOUNT / "My Drive").is_dir() and DRIVE_ROOT.is_dir(),
                     str(DRIVE_ROOT)))
results.append(check("Secret",
                     bool(STATE["secret"]) and valid_encryption_key(STATE["encryption_key"]),
                     "%s = 64 Hex" % SECRET_ENCRYPTION))
results.append(check("FreeLLMAPI Installation",
                     app_ready() and bs3_ok(),
                     "Tag %s, better-sqlite3 ladbar" % FREELLMAPI_REF))
results.append(check("Persistent storage",
                     backup_status(BACKUP_FILE)[1],
                     "%s: %s" % (BACKUP_FILE.name, backup_status(BACKUP_FILE)[0])))

# Verschluesselung: Backup muss den FAPIBK1-Header haben und darf kein SQLite-
# Klartext sein (eine reine SQLite-Datei beginnt mit 'SQLite format 3\0').
enc_ok = False
enc_note = "kein Backup vorhanden"
if backup_status(BACKUP_FILE)[1]:
    with open(BACKUP_FILE, "rb") as f:
        head = f.read(16)
    enc_ok = head.startswith(BACKUP_MAGIC) and not head.startswith(b"SQLite format 3")
    enc_note = "FAPIBK1-Header, AES-256-GCM + gzip, kein SQLite-Klartext"
results.append(check("Encryption configuration", enc_ok, enc_note))

st, _ = http_get("/api/ping", timeout=8)
results.append(check("FreeLLMAPI process", st == 200, "GET /api/ping -> %s" % st))

st, _ = http_get("/v1/openapi.json", timeout=10)
results.append(check("HTTP API", st == 200, "GET /v1/openapi.json -> %s" % st))

if STATE["unified_key"]:
    st, body = http_get("/v1/models", timeout=25,
                        headers={"Authorization": "Bearer " + STATE["unified_key"]})
    n = 0
    if st == 200:
        try:
            n = len(json.loads(body).get("data", []))
        except Exception:
            n = -1
    results.append(check("Model endpoint", st == 200 and n != 0,
                         "GET /v1/models -> %s, %d Eintraege" % (st, max(n, 0))))
else:
    st, _ = http_get("/v1/models", timeout=15)
    r = check("Model endpoint", st == 401,
              "GET /v1/models -> %s (Endpunkt da, Auth aktiv)" % st)
    results.append(r)
    skip("Model endpoint (200)",
         "kein Secret '%s' gesetzt - echter Abruf mit Unified Key nicht moeglich"
         % SECRET_UNIFIEDKEY)

# ------------------- Geheimnis-Suche in Google Drive ------------------------
print("")
info("Suche nach dem ENCRYPTION_KEY in Google Drive ...")
found_in = []
needle = STATE["encryption_key"]
for f in sorted(DRIVE_ROOT.glob("*")):
    if not f.is_file() or fsize(f) > 200 * 1048576:
        continue
    try:
        data = f.read_bytes()
    except OSError:
        continue
    if needle.encode() in data:
        found_in.append(f.name)
results.append(check("Kein Key in Drive", not found_in,
                     "Klartext-Suche in %d Dateien: %s"
                     % (len(list(DRIVE_ROOT.glob('*'))),
                        "nicht gefunden" if not found_in else "GEFUNDEN in " + ", ".join(found_in))))

env_files = list(DRIVE_ROOT.glob("*.env*")) + list(DRIVE_ROOT.glob(".env"))
results.append(check("Keine .env in Drive", not env_files,
                     "gefunden: " + (", ".join(x.name for x in env_files) if env_files else "keine")))

print("")
print("=" * 64)
good = sum(1 for r in results if r is True)
bad  = sum(1 for r in results if r is False)
print("  %d von %d Pruefungen bestanden%s" % (good, len(results),
                                               "" if bad == 0 else ", %d fehlgeschlagen" % bad))
if bad == 0:
    print("")
    print("  FreeLLMAPI laeuft zuverlaessig in Colab und ueberlebt den Neustart")
    print("  mit seinen verschluesselten Daten in Google Drive.")
else:
    print("")
    print("  Nicht alles gruen — die fehlerhafte Zeile oben nennt den Grund.")
print("=" * 64)


## Zelle 8 — Stoppen

Freiwilliges Beenden. Vor dem Ende schreibt FreeLLMAPI kein separates Abschluss-Backup —
die Backup-Pumpe läuft im Takt von `FREEAPI_DB_BACKUP_INTERVAL_MS` (hier 2 Minuten). Wer auf
Nummer sicher gehen will, wartet nach der letzten Änderung kurz, bis im Server-Log wieder
`[db-backup] uploaded …` erscheint, und stoppt dann.

In [ ]:
# ============================================================================
#  ZELLE 8 — STOPPEN
# ============================================================================
print("=" * 64)
print(" FREE LLM API STOPPEN")
print("=" * 64)

# Letztes Backup abwarten, falls juenger als das Intervall?
tail_before = log_text(1500)
info("Letzter Backup-Eintrag im Log:")
lines = [l for l in tail_before.splitlines() if "[db-backup] uploaded" in l]
print("           " + (lines[-1] if lines else "(noch kein Backup im Log dieser Sitzung)"))

stop_server()

print("")
info("Datenstand in Google Drive:")
print("           %s : %s" % (BACKUP_FILE.name, backup_status(BACKUP_FILE)[0]))
print("           %s : %s" % (BACKUP_PREV.name, backup_status(BACKUP_PREV)[0]))
print("")
print("  Die Colab-VM darf jetzt beendet werden.")
print("  Beim naechsten Start: Zellen 1 bis 7 der Reihe nach ausfuehren.")


---

## Kurzüberblick

**1. Einmalig vorbereiten**
Colab-Secret `FREELLMAPI_ENCRYPTION_KEY` mit 64 Hex-Zeichen (`openssl rand -hex 32`) anlegen,
„Notebook access" aktivieren. Mehr nicht.

**2. Welches Secret**
`FREELLMAPI_ENCRYPTION_KEY` (Pflicht). Optional `FREELLMAPI_DB_BACKUP_KEY` und
`FREELLMAPI_UNIFIED_KEY`.

**3. Erster Start**
Zellen 1–7 ausführen → Dashboard → Account anlegen → **Keys** → Provider-Keys eintragen.

**4. Späterer Start**
Neue Colab-VM, dieselben Zellen 1–7. Zelle 4 erkennt das Backup, FreeLLMAPI spielt es beim
Start automatisch ein (`[db-backup] restored … bytes` im Log). Account, Provider-Keys und
Fallback-Ketten sind wieder da.

**5. Wo die persistenten Daten liegen**
`Google Drive/FreeLLMAPI/` — `freeapi.db.backup` (verschlüsselte SQLite),
`freeapi.db.backup.prev` (Rollback), `app-cache-v0.9.5.tar.gz` (nur Code).

**6. Was niemals im Klartext in Drive liegt**
Der `ENCRYPTION_KEY` (lebt nur im Colab-Secret und im Prozess-Speicher), Provider-Keys
(AES-256-GCM in der SQLite), die `.env` (wird gar nicht geschrieben), das Server-Log
(bleibt in `/content`). Zelle 7 sucht aktiv nach dem Schlüssel in Drive.

**7. Woran du erkennst, dass es funktioniert**
`GET /api/ping` → `{"status":"ok",…}`, `/v1/models` → 401 ohne bzw. 200 mit Unified Key,
und im Log `[db-backup] uploaded … bytes to /content/drive/My Drive/FreeLLMAPI/freeapi.db.backup`.

---

### Ehrliche Einschränkungen

- **Colab ist kein Hosting.** Die Colab-FAQ verbietet Web-Dienste „not related to interactive
  compute" und auf der kostenlosen Stufe „bypassing the notebook UI to interact primarily via
  a web UI". Runtimes können jederzeit beendet werden; kostenlose Notebooks laufen maximal
  12 Stunden. Für Dauerbetrieb: eigener Host mit Docker.
- **Datenverlustfenster.** Zwischen zwei Backups (hier 2 Minuten) liegt ein Restrisiko. Bei
  einem abrupten VM-Ende sind die letzten Änderungen bis zum letzten `[db-backup] uploaded`
  weg.
- **`ENCRYPTION_KEY` verloren = Daten verloren.** Es gibt laut
  `docs/env/02-security-and-keys.md` keinen Wiederherstellungsweg. Schlüssel extern sichern.
- **Katalog-Update braucht Internet.** Der Router zieht seinen Modell-Katalog von
  `api.freellmapi.co`. Ohne Netz bleibt der letzte Stand erhalten.
